# 03-3. RandomForest モデル改善版（Databricks Community Edition）

## リクルート飲食店来客数予測コンペティション

**目的**: 03-3で発生した過学習（train 0.369 / valid 0.512、比率0.721）を改善する。

### 過学習の原因と対策

| 項目 | 現行（03-3） | 改善版（本ノートブック） |
|------|-------------|----------------------|
| NaN埋め | `fillna(-999)` | **中央値埋め + 欠損フラグ** |
| 問題点 | -999で不自然な木の分割が発生し、学習データの欠損パターンを丸暗記 | 中央値は分布に馴染み、フラグで欠損情報を保持 |
| 過学習度合い | 0.721 | 改善を期待 |

### 本ノートブックの構成
1. 環境セットアップ（Databricks用）
2. データ読み込み（DBFSパスから）
3. 予約特徴量構築
4. NaN埋め関数（改善版: 中央値 + 欠損フラグ）
5. Single Split 学習
6. 5-fold CV
7. ルールベースベースライン比較
8. 特徴量重要度分析（NaNフラグの重要度も確認）
9. Optunaチューニング（max_depth上限20に制限）
10. 既存モデルとの比較・アンサンブル効果
11. 結果保存

---
## 1. 環境セットアップ（Databricks用）

### データアップロード手順

Databricks Community Edition の DBFS にアップロードが必要なファイル:

| ファイル | サイズ | 用途 |
|---------|-------|------|
| `intermediate/02_feature_design.pkl` | ~219 MB | 特徴量・CV設定・confirmed_settings |
| `intermediate/03-1_lgbm_results.pkl` | ~2 MB | LightGBM結果（比較用） |
| `intermediate/03-2_xgb_results.pkl` | ~1.3 MB | XGBoost結果（比較用） |
| `input/hpg_reserve.csv` | ~130 MB | HPG予約データ |
| `input/air_reserve.csv` | ~4 MB | Air予約データ |
| `input/hpg_store_info.csv` | ~0.3 MB | HPG店舗情報 |
| `input/air_store_info.csv` | ~0.03 MB | Air店舗情報 |
| `input/store_id_relation.csv` | ~0.01 MB | 店舗ID対応表 |

**アップロード方法:**
1. Databricks ワークスペース → Data → DBFS → FileStore
2. Upload ボタンでファイルをアップロード
3. パスは `/FileStore/recruit/` 以下に配置

In [ ]:
# Optunaのインストール（Databricks CE にはデフォルトで入っていない）
%pip install optuna --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12
plt.rcParams['figure.facecolor'] = 'white'

# Databricks CE ではMS Gothicが無い場合がある → フォールバック
try:
    plt.rcParams['font.family'] = 'MS Gothic'
    fig_test = plt.figure(); plt.text(0.5, 0.5, 'テスト'); plt.close()
except:
    plt.rcParams['font.family'] = 'DejaVu Sans'
    print('注意: MS Gothic不可。日本語が□になる場合があります')

SEED = 42
np.random.seed(SEED)

# === Databricks DBFS パス設定 ===
# Databricks CE では /dbfs/ プレフィックスでDBFSにアクセス
DBFS_BASE = Path('/dbfs/FileStore/recruit')
INTERMEDIATE_DIR = DBFS_BASE / 'intermediate'
INPUT_DIR = DBFS_BASE / 'input'
OUTPUT_DIR = DBFS_BASE / 'output'

# ローカル実行時のフォールバック
if not DBFS_BASE.exists():
    print('DBFS パスが見つかりません。ローカルパスにフォールバックします。')
    INTERMEDIATE_DIR = Path('./intermediate')
    INPUT_DIR = Path('../../input')
    OUTPUT_DIR = Path('./intermediate')

def rmsle(y_true, y_pred):
    """RMSLE計算（来客数は1人以上にクリップ）"""
    y_pred = np.clip(y_pred, 1, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

print(f'INTERMEDIATE_DIR: {INTERMEDIATE_DIR}')
print(f'INPUT_DIR: {INPUT_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

---
## 2. データ読み込み（DBFSパスから）

In [ ]:
# 02の中間データ読み込み
with open(INTERMEDIATE_DIR / '02_feature_design.pkl', 'rb') as f:
    prev_02 = pickle.load(f)

# 確定済み設定の読み込み
confirmed = prev_02['confirmed_settings']
TRAIN_START = confirmed['best_train_start']
NAN_STRATEGY = confirmed['best_nan_strategy']
ROLLING_CONFIG = confirmed['best_rolling_config']
TRAIN_PERIOD = confirmed['best_train_period']

# バリデーションfold情報の読み込み
val_folds = prev_02['val_folds']

train_df = prev_02['train_features']
valid_df = prev_02['valid_features']
all_features = prev_02['feature_columns']['all_features']
VALID_START = prev_02['VALID_START']

# TRAIN_STARTフィルタの適用
if TRAIN_START is not None:
    train_df = train_df[train_df['visit_date'] >= TRAIN_START].reset_index(drop=True)

# 先行モデルの結果読み込み
with open(INTERMEDIATE_DIR / '03-1_lgbm_results.pkl', 'rb') as f:
    prev_lgbm = pickle.load(f)
with open(INTERMEDIATE_DIR / '03-2_xgb_results.pkl', 'rb') as f:
    prev_xgb = pickle.load(f)

print('=== 確定済み設定 ===')
for k, v in confirmed.items():
    print(f'  {k}: {v}')
print(f'\nバリデーションfold数: {len(val_folds)}')
for i, fold in enumerate(val_folds):
    print(f'  Fold {i+1}: {fold["val_start"]}~{fold["val_end"]}')
print(f'\n学習データ: {train_df.shape}')
print(f'検証データ: {valid_df.shape}')
print(f'特徴量数: {len(all_features)}')

---
## 3. 予約特徴量構築（03-1と同じ処理）

In [ ]:
# === 予約特徴量の構築（03-1と同じ処理） ===
hpg_res = pd.read_csv(INPUT_DIR / 'hpg_reserve.csv', parse_dates=['visit_datetime', 'reserve_datetime'])
air_res = pd.read_csv(INPUT_DIR / 'air_reserve.csv', parse_dates=['visit_datetime', 'reserve_datetime'])
hpg_store_info = pd.read_csv(INPUT_DIR / 'hpg_store_info.csv')
air_store_info = pd.read_csv(INPUT_DIR / 'air_store_info.csv')
store_rel = pd.read_csv(INPUT_DIR / 'store_id_relation.csv')

hpg_res['visit_date'] = hpg_res['visit_datetime'].dt.normalize()
air_res['visit_date'] = air_res['visit_datetime'].dt.normalize()

hpg_store_info['city'] = hpg_store_info['hpg_area_name'].str.split(' ').str[:2].str.join(' ')
hpg_store_info['prefecture'] = hpg_store_info['hpg_area_name'].str.split(' ').str[0]
hpg_res = hpg_res.merge(hpg_store_info[['hpg_store_id', 'city', 'prefecture']], on='hpg_store_id', how='left')

air_store_info['city'] = air_store_info['air_area_name'].str.split(' ').str[:2].str.join(' ')
air_store_info['prefecture'] = air_store_info['air_area_name'].str.split(' ').str[0]

hpg_city_daily = hpg_res.groupby(['city', 'visit_date']).agg(
    hpg_city_reserve_visitors=('reserve_visitors', 'sum'),
    hpg_city_reserve_count=('reserve_visitors', 'count')).reset_index()
hpg_pref_daily = hpg_res.groupby(['prefecture', 'visit_date']).agg(
    hpg_pref_reserve_visitors=('reserve_visitors', 'sum')).reset_index()
hpg_direct = hpg_res.merge(store_rel, on='hpg_store_id', how='inner')
hpg_store_daily = hpg_direct.groupby(['air_store_id', 'visit_date']).agg(
    hpg_store_reserve_visitors=('reserve_visitors', 'sum')).reset_index()
air_store_daily = air_res.groupby(['air_store_id', 'visit_date']).agg(
    air_reserve_visitors=('reserve_visitors', 'sum'),
    air_reserve_count=('reserve_visitors', 'count')).reset_index()

def add_reserve_features(df):
    df = df.copy()
    store_city = air_store_info.set_index('air_store_id')[['city', 'prefecture']]
    df = df.merge(store_city, left_on='air_store_id', right_index=True, how='left', suffixes=('', '_res'))
    df = df.merge(hpg_city_daily, on=['city', 'visit_date'], how='left')
    df = df.merge(hpg_pref_daily, on=['prefecture', 'visit_date'], how='left')
    df = df.merge(hpg_store_daily, on=['air_store_id', 'visit_date'], how='left')
    df = df.merge(air_store_daily, on=['air_store_id', 'visit_date'], how='left')
    df['total_reserve_visitors'] = df[['hpg_store_reserve_visitors', 'air_reserve_visitors']].sum(axis=1, min_count=1)
    reserve_cols = ['hpg_city_reserve_visitors', 'hpg_city_reserve_count',
                    'hpg_pref_reserve_visitors', 'hpg_store_reserve_visitors',
                    'air_reserve_visitors', 'air_reserve_count', 'total_reserve_visitors']
    for col in reserve_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    df.drop(columns=['city', 'prefecture'], errors='ignore', inplace=True)
    for c in [x for x in df.columns if x.endswith('_res')]:
        df.drop(columns=[c], errors='ignore', inplace=True)
    return df

train_df = add_reserve_features(train_df)
valid_df = add_reserve_features(valid_df)

reserve_features = ['hpg_city_reserve_visitors', 'hpg_city_reserve_count',
                    'hpg_pref_reserve_visitors', 'hpg_store_reserve_visitors',
                    'air_reserve_visitors', 'air_reserve_count', 'total_reserve_visitors']
all_features_v2 = all_features + reserve_features

print(f'予約特徴量追加完了: {len(all_features)} → {len(all_features_v2)}（+{len(reserve_features)}）')
print(f'学習データ: {train_df.shape}')
print(f'検証データ: {valid_df.shape}')
print(f'\n参考スコア:')
print(f'  LightGBM RMSLE: {prev_lgbm["score_single"]:.5f}')
print(f'  XGBoost  RMSLE: {prev_xgb["score_single"]:.5f}')
print(f'  旧RF(fillna-999) RMSLE: 0.51215')

---
## 4. NaN埋め関数（改善版: 中央値 + 欠損フラグ）

### なぜ `fillna(-999)` が問題なのか

RandomForestの決定木は、各ノードで `feature <= threshold` の分割を探索する。

- `-999`は全特徴量の分布から極端に離れた値
- 木は「feature <= -998」のような分割を作り、**NaNだったサンプルだけ**を完璧に分離できる
- 学習データでは「NaNだった箇所」と「来客数」に偶然の相関がある → **丸暗記**
- 検証データでは欠損パターンが異なる → **汎化しない**

### 改善版: 中央値埋め + 欠損フラグ

- **中央値**: 特徴量の分布の中心 → 分割に不自然な影響を与えない
- **欠損フラグ**: `is_nan_XXX = 1/0` → 「欠損だった」情報を明示的に保持
- **リーク防止**: 検証データにも**学習データ**の中央値を使用

In [ ]:
def impute_with_median_and_flags(X_train, X_valid, features):
    """NaN非対応モデル用: 中央値埋め + 欠損フラグ列の追加

    Parameters:
        X_train: 学習データの特徴量DataFrame
        X_valid: 検証データの特徴量DataFrame
        features: 元の特徴量リスト

    Returns:
        X_train, X_valid, nan_flag_cols: 処理済みDataFrameと追加されたフラグ列名リスト
    """
    X_train = X_train.copy()
    X_valid = X_valid.copy()

    # 学習データの中央値を計算（検証データにも同じ値を使う → リーク防止）
    medians = X_train[features].median()

    nan_flag_cols = []
    for col in features:
        if X_train[col].isna().any() or X_valid[col].isna().any():
            flag_col = f'is_nan_{col}'
            X_train[flag_col] = X_train[col].isna().astype(int)
            X_valid[flag_col] = X_valid[col].isna().astype(int)
            nan_flag_cols.append(flag_col)
        X_train[col] = X_train[col].fillna(medians[col])
        X_valid[col] = X_valid[col].fillna(medians[col])

    return X_train, X_valid, nan_flag_cols


# === 動作確認: NaN統計 ===
train_nan_count = train_df[all_features_v2].isna().sum().sum()
valid_nan_count = valid_df[all_features_v2].isna().sum().sum()
nan_features = train_df[all_features_v2].isna().any()
nan_feature_names = nan_features[nan_features].index.tolist()

print(f'=== NaN統計 ===')
print(f'  学習データのNaN数: {train_nan_count:,}')
print(f'  検証データのNaN数: {valid_nan_count:,}')
print(f'  NaNを含む特徴量数: {len(nan_feature_names)}')
print(f'\n  NaNを含む特徴量:')
for col in nan_feature_names:
    train_rate = train_df[col].isna().mean() * 100
    valid_rate = valid_df[col].isna().mean() * 100
    print(f'    {col}: train {train_rate:.1f}%, valid {valid_rate:.1f}%')

---
## 5. Single Split 学習（改善版 vs 旧版の比較）

ハイパーパラメータは旧版と同じ設定で、NaN埋め戦略のみ変更して効果を測定する。

In [ ]:
# === ハイパーパラメータ設定（旧版と同一） ===
rf_params = {
    'n_estimators': 500,
    'max_depth': 20,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'max_features': 0.8,
    'max_samples': 0.8,
    'random_state': SEED,
    'n_jobs': -1,
}

# === 改善版NaN埋めの適用 ===
X_train_imp, X_valid_imp, nan_flag_cols = impute_with_median_and_flags(
    train_df[all_features_v2], valid_df[all_features_v2], all_features_v2
)
all_features_v2_with_flags = all_features_v2 + nan_flag_cols

print(f'=== NaN埋め結果 ===')
print(f'  元の特徴量数: {len(all_features_v2)}')
print(f'  追加フラグ数: {len(nan_flag_cols)}')
print(f'  合計特徴量数: {len(all_features_v2_with_flags)}')
print(f'  埋め込み後NaN数: {X_train_imp[all_features_v2_with_flags].isna().sum().sum()}')

y_train = np.log1p(train_df['visitors'])
y_valid = np.log1p(valid_df['visitors'])

# === Single Split 学習 ===
model = RandomForestRegressor(**rf_params)
model.fit(X_train_imp[all_features_v2_with_flags], y_train)

pred_log = model.predict(X_valid_imp[all_features_v2_with_flags])
pred = np.expm1(pred_log)
score_single = rmsle(valid_df['visitors'], pred)

# 学習データのスコア（過学習チェック）
train_pred = np.expm1(model.predict(X_train_imp[all_features_v2_with_flags]))
score_train = rmsle(train_df['visitors'], train_pred)
overfit_ratio = score_train / score_single

print(f'\n=== Single Split 結果（改善版: 中央値埋め + 欠損フラグ） ===')
print(f'  学習 RMSLE: {score_train:.5f}')
print(f'  検証 RMSLE: {score_single:.5f}')
print(f'  過学習度合い: {overfit_ratio:.3f} (1.0に近いほど良い)')

print(f'\n=== 旧版(fillna-999)との比較 ===')
print(f'  旧版 検証RMSLE: 0.51215 (過学習度合い: 0.721)')
print(f'  改善版 検証RMSLE: {score_single:.5f} (過学習度合い: {overfit_ratio:.3f})')
diff = 0.51215 - score_single
print(f'  改善幅: {diff:+.5f} ({"改善" if diff > 0 else "悪化"})')

print(f'\n=== 他モデルとの比較 ===')
print(f'  LightGBM RMSLE: {prev_lgbm["score_single"]:.5f}')
print(f'  XGBoost  RMSLE: {prev_xgb["score_single"]:.5f}')
print(f'  RF改善版 RMSLE: {score_single:.5f}')

---
## 6. 5-fold CV（val_foldsベースの時系列CV）

各foldで**独立に中央値を計算**してリークを防止する。

In [ ]:
# === val_folds ベース Cross-Validation (5-fold) ===
full_df = pd.concat([train_df, valid_df], ignore_index=True).sort_values('visit_date').reset_index(drop=True)

cv_scores = []
cv_overfit_ratios = []

for fold_i, fold in enumerate(val_folds, 1):
    val_start = pd.Timestamp(fold['val_start'])
    val_end = pd.Timestamp(fold['val_end'])

    train_mask = full_df['visit_date'] < val_start
    valid_mask = (full_df['visit_date'] >= val_start) & (full_df['visit_date'] <= val_end)

    fold_train = full_df[train_mask]
    fold_valid = full_df[valid_mask]

    if len(fold_train) == 0 or len(fold_valid) == 0:
        print(f'Fold {fold_i}: スキップ（データ不足）')
        continue

    # 各foldで独立に中央値埋め + フラグ（リーク防止）
    X_tr, X_va, fold_nan_flags = impute_with_median_and_flags(
        fold_train[all_features_v2], fold_valid[all_features_v2], all_features_v2
    )
    fold_features = all_features_v2 + fold_nan_flags

    y_tr = np.log1p(fold_train['visitors'])
    y_va_raw = fold_valid['visitors']

    m = RandomForestRegressor(**rf_params)
    m.fit(X_tr[fold_features], y_tr)

    # 検証スコア
    p = np.expm1(m.predict(X_va[fold_features]))
    s = rmsle(y_va_raw, p)
    cv_scores.append(s)

    # 過学習チェック
    train_p = np.expm1(m.predict(X_tr[fold_features]))
    s_train = rmsle(fold_train['visitors'], train_p)
    ratio = s_train / s
    cv_overfit_ratios.append(ratio)

    print(f'Fold {fold_i}: RMSLE={s:.5f} (train={s_train:.5f}, ratio={ratio:.3f})  '
          f'n_train={len(fold_train):,}, n_valid={len(fold_valid):,}, '
          f'n_features={len(fold_features)}')

print(f'\n=== CV結果（改善版: 中央値埋め + 欠損フラグ） ===')
print(f'  平均 RMSLE: {np.mean(cv_scores):.5f} (+/- {np.std(cv_scores):.5f})')
print(f'  平均過学習度合い: {np.mean(cv_overfit_ratios):.3f}')

print(f'\n=== 旧版(fillna-999)との比較 ===')
print(f'  旧版 CV平均: 0.51057 (+/- 0.01272), 過学習度合い: 0.721')
print(f'  改善版 CV平均: {np.mean(cv_scores):.5f} (+/- {np.std(cv_scores):.5f}), '
      f'過学習度合い: {np.mean(cv_overfit_ratios):.3f}')

print(f'\n=== 他モデルとの比較 ===')
print(f'  LightGBM CV平均: {prev_lgbm["cv_mean"]:.5f} (+/- {prev_lgbm["cv_std"]:.5f})')
print(f'  XGBoost  CV平均: {prev_xgb["cv_mean"]:.5f} (+/- {prev_xgb["cv_std"]:.5f})')
print(f'  RF改善版 CV平均: {np.mean(cv_scores):.5f} (+/- {np.std(cv_scores):.5f})')

---
## 7. ルールベースベースライン比較

In [ ]:
# === ルールベースベースライン: 店舗×曜日の中央値 ===
full_baseline_df = pd.concat([train_df, valid_df], ignore_index=True)
full_baseline_df['dow'] = pd.to_datetime(full_baseline_df['visit_date']).dt.dayofweek

train_baseline = full_baseline_df[full_baseline_df['visit_date'] < VALID_START]
store_dow_median = train_baseline.groupby(['air_store_id', 'dow'])['visitors'].median()

valid_baseline_df = full_baseline_df[full_baseline_df['visit_date'] >= VALID_START].copy()
valid_baseline_df['baseline_pred'] = valid_baseline_df.apply(
    lambda r: store_dow_median.get((r['air_store_id'], r['dow']),
              train_baseline['visitors'].median()), axis=1)

baseline_rmsle_val = rmsle(valid_baseline_df['visitors'], valid_baseline_df['baseline_pred'])
baseline_pred = valid_baseline_df['baseline_pred'].values

print('=== ルールベースベースライン（店舗×曜日 中央値） ===')
print(f'  ベースライン RMSLE: {baseline_rmsle_val:.5f}')
print(f'  RF改善版    RMSLE: {score_single:.5f}')
print(f'  RF改善幅: {baseline_rmsle_val - score_single:.5f} ({(baseline_rmsle_val - score_single)/baseline_rmsle_val*100:.1f}%)')
print(f'\n  LightGBM     RMSLE: {prev_lgbm["score_single"]:.5f}')
print(f'  XGBoost      RMSLE: {prev_xgb["score_single"]:.5f}')
print(f'  旧RF(-999)   RMSLE: 0.51215 (改善幅: 6.8%)')

---
## 8. 特徴量重要度分析（NaNフラグの重要度も確認）

In [ ]:
importance = pd.DataFrame({
    'feature': all_features_v2_with_flags,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

def categorize_feature(name):
    if name.startswith('is_nan_'):
        return 'NaN欠損フラグ'
    elif name.startswith('rolling_') or name.startswith('ewm_'):
        return 'Rolling統計量'
    elif name.startswith('lag_'):
        return 'ラグ特徴量'
    elif name.startswith(('open_ratio_', 'closed_streak', 'days_since_')):
        return '休業パターン'
    elif name.startswith(('is_after_holiday_', 'holiday_count_', 'is_near_special')):
        return '祝日前後'
    elif name.startswith('store_'):
        return '店舗統計量'
    elif name.startswith('genre_'):
        return 'ジャンル'
    elif name.startswith(('hpg_', 'air_reserve', 'total_reserve')):
        return '予約データ'
    else:
        return '時間/店舗属性'

importance['category'] = importance['feature'].apply(categorize_feature)

color_map = {
    'Rolling統計量': '#DD8452', 'ラグ特徴量': '#55A868',
    '店舗統計量': '#4C72B0', 'ジャンル': '#C44E52', '時間/店舗属性': '#8172B3',
    '休業パターン': '#E5AE38', '祝日前後': '#6ACC65', '予約データ': '#FF6B6B',
    'NaN欠損フラグ': '#999999',
}

fig, axes = plt.subplots(1, 2, figsize=(18, max(10, len(all_features_v2_with_flags) * 0.25)))

colors = importance['category'].map(color_map)
axes[0].barh(importance['feature'], importance['importance'], color=colors)
axes[0].set_xlabel('Importance (Impurity)')
axes[0].set_title(f'RandomForest Feature Importance ({len(all_features_v2_with_flags)} features)')

cat_imp = importance.groupby('category')['importance'].sum().sort_values()
cat_colors = [color_map.get(c, 'gray') for c in cat_imp.index]
cat_imp.plot(kind='barh', ax=axes[1], color=cat_colors)
axes[1].set_xlabel('Importance Sum')
axes[1].set_title('Category Importance')

plt.tight_layout()
plt.show()

# NaNフラグの重要度を分析
total_imp = importance['importance'].sum()
nan_flag_imp = importance[importance['category'] == 'NaN欠損フラグ']
nan_flag_share = nan_flag_imp['importance'].sum() / total_imp * 100

print(f'\n=== NaN欠損フラグの重要度 ===')
print(f'  フラグ数: {len(nan_flag_imp)}')
print(f'  重要度合計シェア: {nan_flag_share:.1f}%')
if len(nan_flag_imp) > 0:
    print(f'\n  個別フラグ重要度:')
    for _, row in nan_flag_imp.sort_values('importance', ascending=False).iterrows():
        pct = row['importance'] / total_imp * 100
        print(f'    {row["feature"]}: {row["importance"]:.4f} ({pct:.1f}%)')

print(f'\n=== 上位10特徴量 ===')
for _, row in importance.tail(10).iloc[::-1].iterrows():
    pct = row['importance'] / total_imp * 100
    marker = ' [NaNフラグ]' if row['category'] == 'NaN欠損フラグ' else ''
    marker = ' [予約]' if row['category'] == '予約データ' else marker
    print(f'  [{row["category"]:10s}] {row["feature"]}: {row["importance"]:.4f} ({pct:.1f}%){marker}')

---
## 9. Optunaチューニング（max_depth上限20に制限）

### 旧版からの変更点
- `max_depth`: 上限を30 → **20**に制限（過学習抑制）
- NaN埋め: 各trial内でもfoldごとに中央値を独立計算
- trial数: 30（RFは学習が遅いため）

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800, step=100),
        'max_depth': trial.suggest_int('max_depth', 8, 20),  # 上限20に制限（旧版は30）
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 20),
        'max_features': trial.suggest_float('max_features', 0.5, 1.0),
        'max_samples': 0.8,
        'random_state': SEED,
        'n_jobs': -1,
    }

    fold_scores = []
    for fold in val_folds:
        val_start = pd.Timestamp(fold['val_start'])
        val_end = pd.Timestamp(fold['val_end'])

        train_mask = full_df['visit_date'] < val_start
        valid_mask = (full_df['visit_date'] >= val_start) & (full_df['visit_date'] <= val_end)

        fold_train = full_df[train_mask]
        fold_valid = full_df[valid_mask]

        if len(fold_train) == 0 or len(fold_valid) == 0:
            continue

        # 各foldで独立に中央値埋め + フラグ
        X_tr, X_va, fold_nan_flags = impute_with_median_and_flags(
            fold_train[all_features_v2], fold_valid[all_features_v2], all_features_v2
        )
        fold_features = all_features_v2 + fold_nan_flags

        y_tr = np.log1p(fold_train['visitors'])
        y_va_raw = fold_valid['visitors']

        m = RandomForestRegressor(**params)
        m.fit(X_tr[fold_features], y_tr)

        p = np.expm1(m.predict(X_va[fold_features]))
        s = rmsle(y_va_raw, p)
        fold_scores.append(s)

    return np.mean(fold_scores)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'=== Optuna チューニング結果（{N_TRIALS} trials, {len(val_folds)}-fold CV） ===')
print(f'  ベストCV RMSLE: {study.best_value:.5f}')
print(f'  チューニング前CV RMSLE: {np.mean(cv_scores):.5f}')
print(f'  改善幅: {np.mean(cv_scores) - study.best_value:.5f}')
print(f'\n  ベストパラメータ:')
for k, v in study.best_params.items():
    print(f'    {k}: {v}')
print(f'\n  旧版Optuna結果: CV 0.50892 (max_depth=24)')
print(f'  改善版Optuna結果: CV {study.best_value:.5f} (max_depth={study.best_params["max_depth"]})')

In [ ]:
# === ベストパラメータでSingle Splitの再学習（予測値保存用） ===
best_rf_params = {
    **study.best_params,
    'max_samples': 0.8,
    'random_state': SEED,
    'n_jobs': -1,
}

model_tuned = RandomForestRegressor(**best_rf_params)
model_tuned.fit(X_train_imp[all_features_v2_with_flags], y_train)

pred_tuned_log = model_tuned.predict(X_valid_imp[all_features_v2_with_flags])
pred_tuned = np.expm1(pred_tuned_log)
score_tuned = rmsle(valid_df['visitors'], pred_tuned)
residuals_tuned = np.log1p(valid_df['visitors'].values) - pred_tuned_log

# 過学習チェック
train_pred_tuned = np.expm1(model_tuned.predict(X_train_imp[all_features_v2_with_flags]))
score_train_tuned = rmsle(train_df['visitors'], train_pred_tuned)
overfit_ratio_tuned = score_train_tuned / score_tuned

print(f'=== チューニング後モデル（Single Split） ===')
print(f'  学習 RMSLE: {score_train_tuned:.5f}')
print(f'  検証 RMSLE: {score_tuned:.5f}')
print(f'  過学習度合い: {overfit_ratio_tuned:.3f}')
print(f'\n=== 全バリエーション比較 ===')
print(f'  旧RF(fillna-999):          0.51215 (過学習: 0.721)')
print(f'  旧RF(fillna-999, tuned):   0.50698')
print(f'  改善RF(中央値+フラグ):      {score_single:.5f} (過学習: {overfit_ratio:.3f})')
print(f'  改善RF(中央値+フラグ,tuned): {score_tuned:.5f} (過学習: {overfit_ratio_tuned:.3f})')
print(f'  LightGBM:                   {prev_lgbm["score_single"]:.5f}')
print(f'  XGBoost:                    {prev_xgb["score_single"]:.5f}')

---
## 10. 既存モデルとの比較・アンサンブル効果

In [ ]:
# === 予測値の相関分析 ===
residuals = np.log1p(valid_df['visitors'].values) - pred_log

pred_comparison = pd.DataFrame({
    'LightGBM': prev_lgbm['valid_pred'],
    'XGBoost': prev_xgb['valid_pred'],
    'RF_improved': pred,
    'RF_tuned': pred_tuned,
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr_matrix = pred_comparison.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.4f', cmap='YlOrRd',
            ax=axes[0], square=True, vmin=0.9, vmax=1.0)
axes[0].set_title('Prediction Correlation')

resid_comparison = pd.DataFrame({
    'LightGBM': prev_lgbm['residuals'],
    'XGBoost': prev_xgb['residuals'],
    'RF_improved': residuals,
    'RF_tuned': residuals_tuned,
})
resid_corr = resid_comparison.corr()
sns.heatmap(resid_corr, annot=True, fmt='.4f', cmap='YlOrRd',
            ax=axes[1], square=True, vmin=0.5, vmax=1.0)
axes[1].set_title('Residual Correlation')

plt.tight_layout()
plt.show()

# === アンサンブル効果 ===
ens_lgb_xgb = (prev_lgbm['valid_pred'] + prev_xgb['valid_pred']) / 2
ens_lgb_xgb_rf = (prev_lgbm['valid_pred'] + prev_xgb['valid_pred'] + pred) / 3
ens_lgb_xgb_rf_tuned = (prev_lgbm['valid_pred'] + prev_xgb['valid_pred'] + pred_tuned) / 3

score_ens2 = rmsle(valid_df['visitors'], ens_lgb_xgb)
score_ens3 = rmsle(valid_df['visitors'], ens_lgb_xgb_rf)
score_ens3_tuned = rmsle(valid_df['visitors'], ens_lgb_xgb_rf_tuned)

print(f'=== アンサンブル効果（単純平均） ===')
print(f'  LGB + XGB:                    {score_ens2:.5f}')
print(f'  LGB + XGB + RF(改善):         {score_ens3:.5f}')
print(f'  LGB + XGB + RF(改善,tuned):   {score_ens3_tuned:.5f}')
print(f'\n  旧版アンサンブル:')
print(f'  LGB + XGB + RF(旧版):         0.48993')
print(f'  LGB + XGB + RF(旧版,tuned):   0.48971')

print(f'\n=== モデル比較サマリ ===')
print(f'{"モデル":<30s} {"Single":>10s} {"CV平均":>10s}')
print('-' * 55)
print(f'{"ベースライン(店舗×DOW)":<30s} {baseline_rmsle_val:>10.5f} {"---":>10s}')
print(f'{"LightGBM":<30s} {prev_lgbm["score_single"]:>10.5f} {prev_lgbm["cv_mean"]:>10.5f}')
print(f'{"XGBoost":<30s} {prev_xgb["score_single"]:>10.5f} {prev_xgb["cv_mean"]:>10.5f}')
print(f'{"RF旧版(fillna-999)":<30s} {"0.51215":>10s} {"0.51057":>10s}')
print(f'{"RF旧版(tuned)":<30s} {"0.50698":>10s} {"0.50892":>10s}')
print(f'{"RF改善版(中央値+フラグ)":<30s} {score_single:>10.5f} {np.mean(cv_scores):>10.5f}')
print(f'{"RF改善版(tuned)":<30s} {score_tuned:>10.5f} {study.best_value:>10.5f}')

---
## 11. 結果保存

結果pklをDBFS（またはローカル）に保存。Databricksの場合はファイルブラウザからダウンロード可能。

In [ ]:
# === 結果の保存 ===
results_03_3 = {
    # デフォルトパラメータ（改善版NaN埋め）
    'valid_pred': pred,
    'valid_pred_log': pred_log,
    'valid_actual': valid_df['visitors'].values,
    'score_single': score_single,
    'cv_scores': cv_scores,
    'cv_mean': np.mean(cv_scores),
    'cv_std': np.std(cv_scores),
    'feature_importance': importance,
    'params': rf_params,
    'residuals': residuals,
    'overfit_ratio': overfit_ratio,
    'cv_overfit_ratios': cv_overfit_ratios,
    # NaN埋め情報
    'nan_strategy': 'median_with_flags',
    'nan_flag_cols': nan_flag_cols,
    'all_features_with_flags': all_features_v2_with_flags,
    # アンサンブル
    'ensemble_scores': {
        'lgb_xgb': score_ens2,
        'lgb_xgb_rf': score_ens3,
    },
    # ベースライン結果
    'baseline_rmsle': baseline_rmsle_val,
    'baseline_pred': baseline_pred,
    'store_dow_median': store_dow_median,
    # チューニング後の結果
    'tuned_params': best_rf_params,
    'tuned_score': score_tuned,
    'tuned_valid_pred': pred_tuned,
    'tuned_valid_pred_log': pred_tuned_log,
    'tuned_residuals': residuals_tuned,
    'tuned_overfit_ratio': overfit_ratio_tuned,
    'tuned_ensemble_scores': {
        'lgb_xgb_rf_tuned': score_ens3_tuned,
    },
    'optuna_best_value': study.best_value,
    'optuna_best_params': study.best_params,
    # 旧版との比較用
    'old_score_single': 0.51215,
    'old_cv_mean': 0.51057,
    'old_overfit_ratio': 0.721,
}

# 出力先ディレクトリの作成
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = OUTPUT_DIR / '03-3_rf_results.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(results_03_3, f)

print(f'結果保存完了: {output_path}')
print(f'  ファイルサイズ: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB')

# モデルの保存
model_path = OUTPUT_DIR / '03-3_rf_model.joblib'
model_tuned_path = OUTPUT_DIR / '03-3_rf_model_tuned.joblib'
joblib.dump(model, model_path)
joblib.dump(model_tuned, model_tuned_path)

print(f'  モデル（デフォルト）: {model_path}')
print(f'  モデル（チューニング後）: {model_tuned_path}')

print(f'\n=== ダウンロード手順（Databricks CE） ===')
print(f'  1. 左サイドバー → Data → DBFS → FileStore → recruit → output')
print(f'  2. 03-3_rf_results.pkl をダウンロード')
print(f'  3. ローカルの intermediate/ に配置')
print(f'  4. 04_比較ノートブックで使用')

---
## まとめ

### 改善のポイント

| 変更点 | 旧版 | 改善版 |
|--------|------|--------|
| NaN埋め | `fillna(-999)` | 中央値埋め + 欠損フラグ |
| max_depth探索範囲 | 10-30 | 8-20 |
| CV内NaN処理 | 全データ共通 | foldごとに独立計算 |

### 検証基準

- 過学習度合い（train/valid比率）が0.721より改善すること
- CV RMSLEが0.51057と同等以上であること
- 結果pklが04_比較ノートブックで読み込めること

### 結果の回収手順

1. Databricksで本ノートブックを最後まで実行
2. DBFS → FileStore → recruit → output から `03-3_rf_results.pkl` をダウンロード
3. ローカルの `notebooks/説明用資料/intermediate/03-3_rf_results.pkl` に上書き配置
4. `04_各モデルの性能比較_20260308.ipynb` を再実行して改善効果を確認